In [1]:
import os
import subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import re
from tqdm import tqdm
import time

In [2]:
fastq_dir = Path("./fastq")
snp_only_dir = Path("./snp_only")
logs_dir = Path("./logs")
vcf_dir = Path("./vcf")

fastq_dir.mkdir(exist_ok=True)
snp_only_dir.mkdir(exist_ok=True)
logs_dir.mkdir(exist_ok=True)
vcf_dir.mkdir(exist_ok=True)

In [3]:
# Group fastq files
samples = {}

for fq in fastq_dir.glob("*.fastq.gz"):
    match = re.search(r'([SE]RR\d+)', fq.name)
    if not match:
        print(f"⚠️ Skipping {fq.name}: no ENA run ID found.")
        continue

    accession = match.group(1)

    if "_1" in fq.name:
        suffix = "_1"
    elif "_2" in fq.name:
        suffix = "_2"
    else:
        continue # if it's not a paired-end, skip

    samples.setdefault(accession, {})[suffix] = fq.resolve()

print(f"✅ Found {len(samples)} samples to process.")

✅ Found 610 samples to process.


In [4]:
# TB-Profiler
# QC, mapping, variant calling, contamination check
def run_tb_profiler(sample, fq_files):
    output_prefix = sample
    tb_cmd = [
        "tb-profiler", "profile",
        "-p", str(output_prefix),
        "--txt"
    ]

    # Ensure correct ordering: -1, -2, -p
    if "_1" in fq_files and "_2" in fq_files:
        tb_cmd = [
            "tb-profiler", "profile",
            "-1", str(fq_files["_1"]),
            "-2", str(fq_files["_2"]),
            "-p", str(output_prefix),
            "--txt"
        ]
    elif "_1" in fq_files:
        tb_cmd = [
            "tb-profiler", "profile",
            "-1", str(fq_files["_1"]),
            "-p", str(output_prefix),
            "--txt"
        ]
    else:
        raise ValueError(f"Missing FASTQ files for {sample}")

    # Write output to log file
    with open(logs_dir / f"{sample}_tbprofiler.log", "w") as log_file:
        subprocess.run(tb_cmd, stdout=log_file, stderr=log_file, check=True)

In [5]:
# Extract and Filter SNPs
def extract_and_filter_snps(sample):
    input_vcf = vcf_dir / f"{sample}.targets.vcf.gz"
    filtered_vcf = snp_only_dir / f"{sample}_snps_only_filtered.vcf"

    bcftools_cmd = [
        "bash", "-c",
        f"bcftools view -v snps {input_vcf} | "
        f"bcftools filter -i 'DP>=5 && QUAL>=20 && AF>=0.75 && SAF>=1 && SAR>=1' "
        f"-o {filtered_vcf} --output-type v"
    ]

    with open(logs_dir / f"{sample}_bcftools_combined.log", "w") as log_file:
        subprocess.run(bcftools_cmd, stdout=log_file, stderr=log_file, check=True)

    return filtered_vcf

In [6]:
# run the vcf first
# run tbprofiler later
def process_sample(sample, fq_files):
    try:
        print(f"🚀 Starting {sample}")
        run_tb_profiler(sample, fq_files)
        filtered_vcf = extract_and_filter_snps(sample)
        return f"✅ {sample} done: {filtered_vcf.name}"
    except Exception as e:
        return f"❌ {sample} failed: {e}"

In [7]:
num_workers = 4           # adjust based on CPU cores
max_retries = 2           # how many times to retry failed samples
retry_wait = 5           # seconds to wait between retries

def run_with_retries(samples, max_retries, retry_wait):
    remaining = samples.copy()

    for attempt in range(1, max_retries + 1):
        print(f"\n🔁 Retry Attempt {attempt}/{max_retries} | Remaining: {len(remaining)} samples")

        failed_samples = {}

        with ThreadPoolExecutor(max_workers=num_workers) as executor:
            futures = {
                executor.submit(process_sample, sample, files): sample
                for sample, files in remaining.items()
            }

            for future in tqdm(as_completed(futures), total=len(futures), desc="Processing samples"):
                sample = futures[future]
                try:
                    result = future.result()
                    print(result)
                except Exception as e:
                    print(f"❌ Exception: {sample} failed with error: {e}")
                    failed_samples[sample] = remaining[sample]

        if not failed_samples:
            print("✅ All samples processed successfully.")
            break
        else:
            print(f"⚠️ {len(failed_samples)} samples failed. Retrying after {retry_wait} seconds...")
            time.sleep(retry_wait)
            remaining = failed_samples  # only retry the ones that failed
    else:
        print("🚫 Reached maximum retries. Some samples could not be processed.")

In [ ]:
run_with_retries(samples, max_retries, retry_wait)


🔁 Retry Attempt 1/2 | Remaining: 610 samples
🚀 Starting ERR038257
🚀 Starting ERR046771
🚀 Starting ERR046840
🚀 Starting ERR046847


Processing samples:   0%|          | 0/610 [00:00<?, ?it/s]